In [46]:
import sqlite3                                 
import pandas as pd                             
 
con = sqlite3.connect('../data/raw/airtel.db')  
 
def q(sql):                                     
    return pd.read_sql(sql, con)

In [47]:
q("""SELECT * FROM circles;""")

,circle_id,circle_name,region
0,1,Delhi,North
1,2,Punjab,North
2,3,Haryana,North
3,4,Himachal Pradesh,North
4,5,Jammu & Kashmir,North
5,6,Rajasthan,North
6,7,Uttar Pradesh (East),North
7,8,Uttar Pradesh (West),North
8,9,Mumbai,West
9,10,Maharashtra,West


In [48]:
q("""
SELECT count(*)AS total_count,
 count(DISTINCT circle_id) AS distinct_count
 FROM circles;
""")

,total_count,distinct_count
0,22,22


In [49]:
q("""SELECT count(*)
    FROM circles c
    JOIN customers cu
    ON c.circle_id = cu.circle_id
    """)

,count(*)
0,22000


In [50]:
q("""
SELECT c.*, cu.*
FROM circles AS c
JOIN customers AS cu
    ON c.circle_id = cu.circle_id;
""")

,circle_id,circle_name,region,customer_id,activation_date,circle_id,age_band,gender,device_type,os,acquisition_channel,connection_type,current_plan_id,arpu_inr,ab_group,is_churned,churn_date
0,1,Delhi,North,9,2022-09-04,1,25-34,Male,Smartphone,Android,Website,Prepaid,5,374,B,0,NaN
1,1,Delhi,North,31,2024-02-10,1,25-34,Male,Smartphone,iOS,Call Center,Prepaid,8,257,A,0,NaN
2,1,Delhi,North,151,2022-07-22,1,35-44,Female,Smartphone,iOS,Airtel App,Prepaid,5,374,B,0,NaN
3,1,Delhi,North,186,2022-01-04,1,25-34,Male,Smartphone,Android,Retailer,Prepaid,8,257,B,0,NaN
4,1,Delhi,North,207,2024-12-14,1,18-24,Other,Smartphone,Android,Retailer,Prepaid,3,224,A,1,2025-04-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21995,22,North East,East,21805,2022-09-03,22,25-34,Female,Feature Phone,Other,Call Center,Prepaid,7,257,A,0,NaN
21996,22,North East,East,21831,2022-12-18,22,25-34,Female,Smartphone,iOS,Website,Prepaid,8,257,B,0,NaN
21997,22,North East,East,21906,2024-05-16,22,25-34,Male,Smartphone,iOS,Airtel App,Prepaid,6,428,B,0,NaN
21998,22,North East,East,21917,2024-02-25,22,25-34,Female,Feature Phone,Other,Retailer,Prepaid,5,374,B,0,NaN


In [54]:
q("""SELECT SUM(is_churned =1 AND churn_date IS NOT NULL) 
       AS left_dated,
    SUM(is_churned = 0 AND churn_date IS NULL) AS stayed_blank
    from customers;""")

,left_dated,stayed_blank
0,5226,16774


In [55]:
q("""SELECT c.customer_id,c.connection_type,
       (SELECT COUNT(*) FROM recharges r
       where r.customer_id = c.customer_id)
       AS payments
FROM customers c
WHERE NOT EXISTS (SELECT 1 FROM usage_daily u
                  where u.customer_id
                  =c.customer_id);""")

,customer_id,connection_type,payments
0,266,Prepaid,1
1,10684,Prepaid,29
2,13508,Prepaid,8
3,19111,Prepaid,39


In [57]:
q("""WITH l AS (SELECT customer_id,     
             MAX(usage_date) AS last_use         
           FROM usage_daily GROUP BY customer_id)
SELECT c.is_churned, COUNT(*) AS subs, 
       ROUND(AVG(julianday('2025-06-30')
             - julianday(l.last_use)), 2)     
         AS avg_quiet,                    
       MIN(julianday('2025-06-30')              
             - julianday(l.last_use)) AS min_,
       MAX(julianday('2025-06-30')              
             - julianday(l.last_use)) AS max_   
FROM   customers c JOIN l                
       ON l.customer_id = c.customer_id     
GROUP  BY c.is_churned;""")

,is_churned,subs,avg_quiet,min_,max_
0,0,16772,2.66,0.0,143.0
1,1,5224,87.80,7.0,180.0


In [58]:
q("""SELECT 'usage_daily.usage_date' AS col,        
       MIN(usage_date) AS first_,                
       MAX(usage_date) AS last_                
FROM   usage_daily                    
UNION ALL SELECT 'customers.activation_date',   
       MIN(activation_date), MAX(activation_date)
FROM   customers                              
UNION ALL SELECT 'customers.churn_date',
       MIN(churn_date), MAX(churn_date) 
FROM   customers                      
UNION ALL SELECT 'recharges.recharge_date', 
       MIN(recharge_date), MAX(recharge_date)
FROM   recharges                           
UNION ALL SELECT 'complaints.complaint_date',
       MIN(complaint_date), MAX(complaint_date)
FROM   complaints;""")

,col,first_,last_
0,usage_daily.usage_date,2025-01-01,2025-06-30
1,customers.activation_date,2022-01-01,2025-02-28
2,customers.churn_date,2025-01-16,2025-06-23
3,recharges.recharge_date,2022-01-01,2025-06-27
4,complaints.complaint_date,2025-01-01,2025-06-30


In [59]:
q("""WITH t AS (SELECT customer_id,                
             SUM(data_mb) AS total,           
             COUNT(*) AS days_active         
           FROM usage_daily GROUP BY customer_id) 
SELECT ROUND(AVG(total / days_active), 2)      
         AS per_active_day,                    
       ROUND(AVG(total / 181.0), 2)              
         AS per_calendar_day              
FROM   t;""")

,per_active_day,per_calendar_day
0,1404.44,524.15
